<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/03_limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Limpieza del dataset unido

Este notebook aplica las decisiones confirmadas en `02_eda.ipynb` sobre
`dataset_unido_raw.csv` (24,160 filas: 12,080 phishing + 12,080 legítimo).

Orden de limpieza (justificado en el EDA):
1. Corregir encoding y normalizar homóglifos/marcas invisibles
2. Eliminar HTML residual (tags + hidden-span) — máximo impacto en truncamiento
3. Filtrar a inglés real (Ronda 1 de entrenamiento)
4. Eliminar duplicados (exactos + normalizando campos variables)
5. Re-evaluar balance de clases

## 0. Instalación de dependencias y carga de datos

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install ftfy unidecode langdetect -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 11.1 MB/s eta 0:00:00


In [4]:
import pandas as pd

RUTA_DATASET = '/content/drive/MyDrive/dataset_unido_raw.csv'
df = pd.read_csv(RUTA_DATASET)

print('Filas iniciales:', len(df))
print('Columnas:', df.columns.tolist())
print()
print('Distribución de etiquetas:')
print(df['label'].value_counts())

Filas iniciales: 24160
Columnas: ['text', 'label']

Distribución de etiquetas:
label
1    12080
0    12080
Name: count, dtype: int64


## 1. Reconstruir columna 'source'

No quedó guardada en la versión actual de `01_extraccion.ipynb`. Se
reconstruye usando el orden de concatenación original (pot → nazario →
enron) y los conteos conocidos de cada fuente.

In [5]:
N_POT = 8614
N_NAZARIO = 3466
N_ENRON = len(df) - N_POT - N_NAZARIO

df['source'] = (
    ['phishing_pot'] * N_POT +
    ['nazario'] * N_NAZARIO +
    ['enron'] * N_ENRON
)

print('Verificación fuente x etiqueta:')
print(pd.crosstab(df['source'], df['label']))

Verificación fuente x etiqueta:
label             0     1
source                   
enron         12080     0
nazario           0  3466
phishing_pot      0  8614


## 2. Corrección de encoding y normalización de caracteres

Este paso debe ir primero porque corrige la base de texto sobre la que
actúan todos los pasos siguientes (HTML, idioma, duplicados). Cubre:

- Encoding roto (mojibake) con `ftfy`
- Entidades HTML corrompidas por el marcador de anonimización
  `phishing@pot` insertado dentro del código numérico (irrecuperables,
  se eliminan enteras)
- Decodificación de entidades HTML normales
- Transliteración de scripts usados como disfraz visual (cirílico,
  griego, armenio, numerales romanos) — sin tocar acentos legítimos
  del español/portugués (á, é, ñ)
- Eliminación de marcas combinadoras invisibles insertadas entre letras
- Eliminación del marcador `phishing@pot` pegado en palabras normales

In [6]:
import re
import html
import unicodedata
import ftfy
from unidecode import unidecode

COMBINING_MARK_RE = re.compile(
    r"[\u0300-\u036f\u0483-\u0489\u0591-\u05c7\u0610-\u061a\u064b-\u065f"
    r"\u0670\u06d6-\u06dc\u0e31\u0e34-\u0e3a\u0e47-\u0e4e\u20d0-\u20ff\ufe00-\ufe0f]"
)
INVISIBLE_CHARS_RE = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060-\u2064\U000e0100-\U000e01ef]")
LOOKALIKE_SCRIPT_RE = re.compile(r"[\u0370-\u03ff\u0400-\u04ff\u0530-\u058f\u2150-\u2189]")
BROKEN_ENTITY_RE = re.compile(r"&(amp;)?#phishing@pot\d*;?")
ANON_PLACEHOLDER_RE = re.compile(r"phishing@pot,?")


def corregir_encoding_y_caracteres(texto):
    if not isinstance(texto, str):
        return ""

    texto = ftfy.fix_text(texto)
    texto = BROKEN_ENTITY_RE.sub("", texto)
    texto = html.unescape(texto)
    texto = LOOKALIKE_SCRIPT_RE.sub(lambda m: unidecode(m.group(0)), texto)
    texto = COMBINING_MARK_RE.sub("", texto)
    texto = INVISIBLE_CHARS_RE.sub("", texto)
    texto = ANON_PLACEHOLDER_RE.sub("", texto)
    texto = unicodedata.normalize("NFKC", texto)

    return texto


df['text'] = df['text'].apply(corregir_encoding_y_caracteres)
print('Corrección de encoding aplicada a las', len(df), 'filas.')

Corrección de encoding aplicada a las 24160 filas.


## 3. Eliminación de HTML residual y hidden-span

Según el EDA, 35.44% de las filas tiene HTML residual, con correlación
de 0.65 con la longitud total, y las filas con HTML se truncan 77.4% de
las veces (vs 15.1% sin HTML) — este es el paso de mayor impacto en la
reducción de truncamiento de tokens.

Se eliminan: tags HTML completos (incluyendo su contenido si es
"hidden text spam" vía `visibility:hidden`/`font-size:0`), declaraciones
DOCTYPE, y atributos de estilo sueltos.

In [54]:
import re
from bs4 import BeautifulSoup
import warnings
from bs4 import MarkupResemblesLocatorWarning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

# --- Protección de cadenas reales (wallets, hashes) ---
HEX_RE = re.compile(r'^[0-9a-fA-F]+$')
BTC_LEGACY_RE = re.compile(r'^[13][a-km-zA-HJ-NP-Z1-9]{25,34}$')
BTC_BECH32_RE = re.compile(r'^bc1[a-z0-9]{25,39}$', re.IGNORECASE)
LTC_BECH32_RE = re.compile(r'^ltc1[a-z0-9]{25,39}$', re.IGNORECASE)
ETH_RE = re.compile(r'^0x[a-fA-F0-9]{40}$')
PATRON_RELLENO = re.compile(r'\b[A-Za-z0-9]{40,}\b')

# --- Hidden text spam (Bradesco) ---
PATRON_HIDDEN_SPAN = re.compile(
    r'<[a-zA-Z]+[^>]*(?:visibility:\s*hidden|font-size:\s*0)[^>]*>.*?</[a-zA-Z]+>',
    re.IGNORECASE | re.DOTALL
)

# --- CSS: @import, @font-face, comentarios, propiedades sueltas ---
PATRON_IMPORT_CSS = re.compile(r'@import\s+url\([^)]*\)\s*;?', re.IGNORECASE)
PATRON_FONTFACE = re.compile(r'@font-face\s*\{[^}]*\}|@font-face\b', re.IGNORECASE)
PATRON_COMENTARIO_CSS = re.compile(r'/\*.*?\*/', re.DOTALL)
PATRON_CSS_PROP_GENERAL = re.compile(
    r'\b[a-z][a-z-]{2,}\s*:\s*(?:[\w%#.\-]+(?:\s+[\w%#.\-]+){0,4})?\s*(!important)?(?=\s|$|;|\{|\})'
)
PATRON_CSS_SUELTO = re.compile(
    r'@media[^{]*\{[^}]*\}|'
    r'\{[^{}]*\}|'
    r'(\.\w[\w-]*[\s,]*){1,}\}?|'
    r'(\b(h[1-6]|table|td|tr|body|div|span|p|a|ul|ol|li|img)\b[\s,]*){3,}|'
    r'#\w[\w-]*(\[[^\]]*\])?|'
    r'\w[\w-]*\[[^\]]*\]|'
    r'\*\[[^\]]*\]|'
    r'\{+|'
    r'\}+'
)


def es_cadena_protegida(cadena):
    if HEX_RE.match(cadena):
        return True
    if BTC_LEGACY_RE.match(cadena):
        return True
    if BTC_BECH32_RE.match(cadena) or LTC_BECH32_RE.match(cadena):
        return True
    if ETH_RE.match(cadena):
        return True
    return False


def limpiar_html_css_relleno(texto):
    texto = str(texto)

    # 1. Hidden-span (antes del parser, para capturar su contenido)
    texto = PATRON_HIDDEN_SPAN.sub(' ', texto)

    # 2. Parser HTML real: descarta <style>/<script> completos
    soup = BeautifulSoup(texto, 'html.parser')
    for tag in soup(['style', 'script']):
        tag.decompose()
    texto = soup.get_text(separator=' ')

    # 3. Comentarios HTML mal cerrados (ej. '-->' huérfano)
    texto = re.sub(r'<!--.*?-->|-->|<!--', ' ', texto, flags=re.DOTALL)

    # 4. Relleno random alfanumérico (protegiendo wallets/hashes)
    def _reemplazar_relleno(m):
        cadena = m.group()
        return cadena if es_cadena_protegida(cadena) else ''
    texto = PATRON_RELLENO.sub(_reemplazar_relleno, texto)

    # 5. CSS suelto que sobrevivió al parser
    texto = PATRON_IMPORT_CSS.sub(' ', texto)
    texto = PATRON_FONTFACE.sub(' ', texto)
    texto = PATRON_COMENTARIO_CSS.sub(' ', texto)
    texto = PATRON_CSS_PROP_GENERAL.sub(' ', texto)

    prev = None
    n_iter = 0
    while prev != texto and n_iter < 10:
        prev = texto
        texto = PATRON_CSS_SUELTO.sub(' ', texto)
        n_iter += 1

    texto = re.sub(r'_{5,}', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


df['text'] = df['text'].apply(limpiar_html_css_relleno)
print('Limpieza consolidada de HTML/hidden-span/relleno/CSS aplicada (v2, casos ampliados).')

df['n_caracteres_post_limpieza'] = df['text'].str.len()
print('Nueva media de longitud:', round(df['n_caracteres_post_limpieza'].mean(), 1))
print('Nuevo percentil 99:', df['n_caracteres_post_limpieza'].quantile(0.99))
print('Nuevo máximo:', df['n_caracteres_post_limpieza'].max())

Limpieza consolidada de HTML/hidden-span/relleno/CSS aplicada (v2, casos ampliados).
Nueva media de longitud: 1357.7
Nuevo percentil 99: 11135.529999999995
Nuevo máximo: 715885


In [55]:
muestra_revision = df.sample(50, random_state=7)

for i, row in muestra_revision.iterrows():
    print(f"=== Fila {i} | label: {row['label']} ===")
    print(row['text'])
    print()
    print("-" * 100)
    print()

=== Fila 14460 | label: 0 ===
FW: Nymex Strip -----Original Message----- From: Germany, Chris Sent: Tuesday, April 23, 2002 9:43 AM To: Knippa, Mark Subject: FW: Nymex Strip -----Original Message----- From: Polsky, Phil Sent: Thursday, April 18, 2002 8:44 AM To: Germany, Chris Subject: Nymex Strip go to the link below to look at the gas strip. -----Original Message----- From: Polsky, Phil Sent: Wednesday, April 17, 2002 4:27 PM To: Gregory, Paul Subject: http://hotcs /fundyplasma/site/Financial

----------------------------------------------------------------------------------------------------

=== Fila 20859 | label: 0 ===
Peoples Energy Project Status Rain, Has Demetrion set up a meeting with you? If not, please call him and arrange a time. Thanks ---------------------- Forwarded by Hunter S Shively/HOU/ECT on 01/09/2001 10:08 AM --------------------------- From: Demetrion Ware@ENRON on 01/04/2001 10:47 AM To: Hunter S Shively/HOU/ECT@ECT, Patrice L Mims/HOU/ECT@ECT, Robert Superty/

## 4. Filtrado a inglés real (Ronda 1 de entrenamiento)

Según el EDA, 11.5% de las filas no están en inglés, concentrado en
phishing_pot (28.98%). Se filtra a inglés real para la Ronda 1
(entrenar y evaluar en inglés) — necesario para que la premisa del
experimento sea coherente.

Para la Ronda 2 (traducción a español), se usará la columna `idioma`
generada aquí, traduciendo cada fila desde su idioma real detectado
(no asumiendo que todo es inglés).

In [56]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detectar_idioma_seguro(texto):
    try:
        return detect(str(texto))
    except Exception:
        return 'desconocido'

df['idioma'] = df['text'].apply(detectar_idioma_seguro)

print('Distribución de idiomas tras la limpieza:')
print(df['idioma'].value_counts().head(10))
print()
print(f"% en inglés: {100 * (df['idioma'] == 'en').mean():.1f}%")

Distribución de idiomas tras la limpieza:
idioma
en             19591
de              1822
pt              1197
fr               632
nl               490
es               118
it                43
desconocido       34
cy                31
no                25
Name: count, dtype: int64

% en inglés: 81.1%


In [57]:
antes = len(df)
df_ingles = df[df['idioma'] == 'en'].copy().reset_index(drop=True)
print(f'Eliminadas {antes - len(df_ingles)} filas no identificadas como inglés.')
print(f'Filas finales (solo inglés): {len(df_ingles)}')
print()
print('Distribución de etiquetas tras filtrar idioma:')
print(df_ingles['label'].value_counts())

Eliminadas 4569 filas no identificadas como inglés.
Filas finales (solo inglés): 19591

Distribución de etiquetas tras filtrar idioma:
label
0    11967
1     7624
Name: count, dtype: int64


## 5. Eliminación de duplicados

Según el EDA: 7.11% de duplicados exactos, +112 adicionales si se
normalizan campos variables (fecha, monto, ID, hora, email del
destinatario) — correos de la misma plantilla de phishing con esos
valores cambiados por destinatario.

Se trabaja sobre `df_ingles` (ya filtrado a inglés), no sobre el
dataset multilingüe completo.

In [59]:
import re

PATRONES_VARIABLES = {
    'FECHA': r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b',
    'MONTO': r'[$€£]\s?[\d.,]+\d',
    'ID_CODIGO': r'\b(ID|Code|Member ID|User ID|Invitation Code|Subscription ID)[:\s#]*[A-Za-z0-9-]{4,}\b',
    'CODIGO_HASH': r'#[A-Z]{2,6}-\d{4,10}\b',  # nuevo: códigos tipo #AJPX-9769160
    'TELEFONO': r'\+?\d{1,3}\s?\(\d{2,4}\)\s?\d{3,4}-\d{4}',  # nuevo: +1 (888) 441-4083
    'HORA_RELATIVA': r'\b\d{1,2}\s?(hours?|hrs?|minutes?|mins?)\b|before\s+midnight|\d{1,2}:\d{2}\s?(AM|PM|am|pm)?',
    'PORCENTAJE': r'\b\d{2,4}%',  # nuevo: 400%, 200%
    'SPINS': r'\b\d{2,4}\s+(FREE\s+SPINS|Freispiele|FS)\b',  # nuevo: 200 FREE SPINS, 400 FS
    'EMAIL_USUARIO': r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b',
    'DIRECCION': r'\b\d{1,6}\s+[A-Za-z][A-Za-z0-9.\s]{2,35}\b(?:Ave|Avenue|St|Street|Rd|Road|Dr|Drive|Blvd|Ln|Lane|Way)\b(?:,?\s+(?:Suite|Ste)\.?\s*\d+)?',
}

def normalizar_para_dedupe(texto):
    texto = str(texto)
    texto = re.sub(
        r'(Final Notice Coming for an?\s+)([A-Za-z0-9\s]{3,40})(\s+Reward)',
        r'\1<PRODUCTO>\3', texto, flags=re.IGNORECASE
    )
    for nombre, patron in PATRONES_VARIABLES.items():
        texto = re.sub(patron, f'<{nombre}>', texto, flags=re.IGNORECASE)
    return texto

df_ingles['text_normalizado_dedupe'] = df_ingles['text'].apply(normalizar_para_dedupe)

n_exactos = df_ingles.duplicated(subset=['text']).sum()
n_plantilla = df_ingles.duplicated(subset=['text_normalizado_dedupe']).sum()

print(f'Duplicados exactos: {n_exactos} ({100*n_exactos/len(df_ingles):.2f}%)')
print(f'Duplicados de plantilla (normalizado): {n_plantilla} ({100*n_plantilla/len(df_ingles):.2f}%)')

Duplicados exactos: 1620 (8.27%)
Duplicados de plantilla (normalizado): 1734 (8.85%)


In [60]:
antes = len(df_ingles)
df_ingles = df_ingles.drop_duplicates(subset=['text_normalizado_dedupe']).reset_index(drop=True)
df_ingles = df_ingles.drop(columns=['text_normalizado_dedupe'])

print(f'Eliminados {antes - len(df_ingles)} duplicados (exactos + plantilla).')
print(f'Filas finales: {len(df_ingles)}')
print()
print('Distribución de etiquetas tras deduplicar:')
print(df_ingles['label'].value_counts())

Eliminados 1734 duplicados (exactos + plantilla).
Filas finales: 17857

Distribución de etiquetas tras deduplicar:
label
0    11509
1     6348
Name: count, dtype: int64


## 8. Filtro de longitud mínima

Según PEEK (Phishing Evolution Framework, 2024), se establece un mínimo
de 64 tokens por correo, para asegurar que cada fila tenga suficiente
contenido real (no solo el asunto sin cuerpo, como los casos vistos en
el EDA: "DHL Delivery", "Birth surgeries + examinations + tests ....").

No se establece un máximo explícito: la mayoría de papers de BERT/RoBERTa
revisados dejan que el propio tokenizer trunque automáticamente
(`truncation=True, max_length=512`) al momento de entrenar, en vez de
recortar el texto de antemano.

In [ ]:
!pip install transformers -q

In [61]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')

df_ingles['n_tokens'] = df_ingles['text'].apply(lambda t: len(tokenizer.tokenize(str(t))))

antes = len(df_ingles)
df_ingles = df_ingles[df_ingles['n_tokens'] >= 64].drop(columns=['n_tokens']).reset_index(drop=True)
print(f'Eliminadas {antes - len(df_ingles)} filas con menos de 64 tokens.')
print(df_ingles['label'].value_counts())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2726 > 512). Running this sequence through the model will result in indexing errors


Eliminadas 3778 filas con menos de 64 tokens.
label
0    9312
1    4767
Name: count, dtype: int64


## 9. Re-balanceo final

Tras el filtro de longitud, el balance de clases vuelve a romperse
(consistente con todos los pasos anteriores: phishing_pot concentra más
filas cortas que Enron). Se submuestrea la clase legítima para igualar
el número final de phishing.

In [62]:
N_PHISHING_FINAL = (df_ingles['label'] == 1).sum()

df_phishing = df_ingles[df_ingles['label'] == 1]
df_legitimo = df_ingles[df_ingles['label'] == 0].sample(n=N_PHISHING_FINAL, random_state=42)

df_final = pd.concat([df_phishing, df_legitimo], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print('Balance final:')
print(df_final['label'].value_counts())

Balance final:
label
1    4767
0    4767
Name: count, dtype: int64


## 10. Exportación final

Se conservan únicamente las columnas `text` y `label`, y se guarda el
dataset final que se usará en `04_traduccion.ipynb` y `05_entrenamiento.ipynb`.

In [63]:
df_final = df_final[['text', 'label']]

ruta_salida = '/content/drive/MyDrive/dataset_limpio_ingles_balanceado.csv'
df_final.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print('Guardado en:', ruta_salida, '| filas:', len(df_final))

Guardado en: /content/drive/MyDrive/dataset_limpio_ingles_balanceado.csv | filas: 9534
